In [1]:
!pip install hmmlearn

In [3]:
# hmm_predict.py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings("ignore")

# ------------------------
# 1. Load & prepare data
# ------------------------
df = pd.read_csv("data.csv")   # replace path if needed
df['Date'] = pd.to_datetime(df['Date'])

# Create next-day price movement (target) and drop last row per symbol safely
df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)
df['Price_Movement'] = df.groupby('Symbol')['Close'].shift(-1) - df['Close']
df.dropna(subset=['Price_Movement'], inplace=True)  # remove rows w/o next-day close

# Build a classification target for evaluation: Up (1) if next-day move > 0, else 0
df['TargetUp'] = (df['Price_Movement'] > 0).astype(int)

# ------------------------
# 2. Choose observation features for HMM (continuous)
# ------------------------
features = [
    'News - Positive Sentiment', 'News - Negative Sentiment',
    'News - Analyst Comments', 'News - Stocks',
    'News - Corporate Earnings', 'Close'
]
# Keep only rows where these exist
df = df.dropna(subset=features)

# ------------------------
# 3. Build sequences per symbol
# ------------------------
# We'll create a list of sequences (each sequence = N_time x n_features),
# and a parallel list of targets (next-day Up/Down) aligned to each time step,
# but for prediction we will forecast the next step after each sequence's last observed row.
sequence_list = []
target_next_list = []   # target (Up/Down) for the next day after the last step in sequence
symbols = []

# Set window length for each sequence. For demonstration keep variable-length sequences:
min_len = 10  # require at least 10 days for a symbol to be included (adjust as needed)

for sym, g in df.groupby('Symbol'):
    g = g.sort_values('Date')
    if len(g) < min_len:
        continue
    # Build one full sequence per symbol (you can split into multiple windows if you want)
    obs = g[features].values.astype(float)
    # The target for predicting the next day after the sequence = last row's TargetUp
    # But careful: our Price_Movement in df already refers to next-day movement,
    # so the "target for last row in sequence" is the last row's TargetUp.
    target_next = g['TargetUp'].values[-1]
    sequence_list.append(obs)
    target_next_list.append(target_next)
    symbols.append(sym)

print(f"Built {len(sequence_list)} sequences from symbols (min_len={min_len}).")

if len(sequence_list) == 0:
    raise RuntimeError("No sequences found. Reduce min_len or check dataset.")

# ------------------------
# 4. Train/test split at sequence level
# ------------------------
seq_train, seq_test, targ_train, targ_test = train_test_split(
    sequence_list, target_next_list, test_size=0.25, random_state=42
)

# ------------------------
# 5. Standardize features across all sequences (fit on train only)
# ------------------------
# Flatten train sequences to fit scaler
concat_train = np.vstack(seq_train)
scaler = StandardScaler().fit(concat_train)

def scale_seq_list(seq_list):
    out = []
    for s in seq_list:
        out.append(scaler.transform(s))
    return out

seq_train_scaled = scale_seq_list(seq_train)
seq_test_scaled = scale_seq_list(seq_test)

# ------------------------
# 6. Prepare data for hmmlearn (concatenate sequences and keep lengths)
# ------------------------
X_train_concat = np.vstack(seq_train_scaled)
lengths_train = [len(s) for s in seq_train_scaled]

# ------------------------
# 7. Fit Gaussian HMM
# ------------------------
n_components = 3   # number of hidden states (tune it)
model = GaussianHMM(n_components=n_components, covariance_type='full', n_iter=200, random_state=42)
model.fit(X_train_concat, lengths=lengths_train)
print("HMM trained successfully!")
print("Hidden states:", model.n_components)
print("Means shape:", model.means_.shape)


# Print emission means for each hidden state (for interpretation)
print("State means (emission means) for each hidden state:")
for i, m in enumerate(model.means_):
    print(f" State {i}: {m}")

# ------------------------
# 8. Predict / Forecast next-step for test sequences
# ------------------------
y_true = []
y_pred = []

for seq, true_target in zip(seq_test_scaled, targ_test):
    # Compute the hidden state sequence for the observed seq
    hidden_states = model.predict(seq)
    last_state = hidden_states[-1]
    # Option A: forecast next observation by using the state's mean (expected observation)
    pred_obs = model.means_[last_state]
    # Convert this predicted observation into a predicted price movement direction:
    # We used 'Close' as the last column of features. If predicted mean 'Close' is higher than
    # the last observed Close in sequence, predict Up (1) else Down (0).
    last_obs_close = seq[-1, features.index('Close')] if 'Close' in features else seq[-1, -1]
    predicted_close = pred_obs[features.index('Close')] if 'Close' in features else pred_obs[-1]
    pred_up = int(predicted_close > last_obs_close)
    y_true.append(true_target)
    y_pred.append(pred_up)

# ------------------------
# 9. Evaluation
# ------------------------
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
acc = accuracy_score(y_true, y_pred)
print(f"HMM forecast Up/Down accuracy on test sequences: {acc:.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print("Classification report:")
print(classification_report(y_true, y_pred))

# ------------------------
# 10. Example: show predicted vs true for first 10 test sequences
# ------------------------
for i in range(min(10, len(y_true))):
    print(f"Seq {i}: true_next_up={y_true[i]}, pred_next_up={y_pred[i]}")


Built 495 sequences from symbols (min_len=10).


Model is not converging.  Current: 1292158.842113777 is not greater than 1292158.8529765527. Delta is -0.0108627756126225


HMM trained successfully!
Hidden states: 3
Means shape: (3, 6)
State means (emission means) for each hidden state:
 State 0: [-0.13065576 -0.21624565 -0.17200849 -0.18962997 -0.15408023 -0.09858067]
 State 1: [1.02975701 1.63880622 1.48942732 1.58017276 1.13802511 1.23167118]
 State 2: [ 0.0050067   0.02972419 -0.03716872 -0.02074407  0.03088447 -0.1449935 ]
HMM forecast Up/Down accuracy on test sequences: 0.3226
Confusion matrix:
[[19 62]
 [22 21]]
Classification report:
              precision    recall  f1-score   support

           0       0.46      0.23      0.31        81
           1       0.25      0.49      0.33        43

    accuracy                           0.32       124
   macro avg       0.36      0.36      0.32       124
weighted avg       0.39      0.32      0.32       124

Seq 0: true_next_up=1, pred_next_up=0
Seq 1: true_next_up=0, pred_next_up=1
Seq 2: true_next_up=0, pred_next_up=1
Seq 3: true_next_up=0, pred_next_up=1
Seq 4: true_next_up=0, pred_next_up=0
Seq 5: